# HW2 - Look at the paintings first

Goal: know the data before modeling it. Answer for the report: How many paintings per split and style? What do
they look like at 64 x 64? How much of a painting does the center crop remove? How are the colors distributed?

Needs the prepared arrays: `uv run python -m dgm.data.wikiart --sizes 64`. Select the `.venv` kernel (top right).

In [ ]:
import csv
import io
import zipfile

import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid

from dgm.data.wikiart import WikiArt, archive_path, describe

SIZE = 64
print(describe(SIZE))
train = WikiArt("train", SIZE)

In [ ]:
# Paintings per style (training split)
counts = np.bincount(train.labels.numpy(), minlength=len(train.style_names))
order = np.argsort(counts)
plt.figure(figsize=(8, 6))
plt.barh([train.style_names[i].replace("_", " ") for i in order], counts[order], color="#2a78d6")
plt.xlabel("paintings (train)")
plt.tight_layout()
plt.show()

In [ ]:
# 8 paintings of 6 styles, as the GAN sees them (64 x 64)
wanted = [
    "Early_Renaissance",
    "Baroque",
    "Impressionism",
    "Cubism",
    "Ukiyo_e",
    "Color_Field_Painting",
]
shown = [s for s in wanted if s in train.style_names] or train.style_names[:6]
fig, axes = plt.subplots(2, 3, figsize=(13, 5), squeeze=False)
for ax, style in zip(axes.flat, shown, strict=False):
    idx = (train.labels == train.style_names.index(style)).nonzero().flatten()[:8]
    grid = make_grid(train.images[idx].float() / 255, nrow=4, padding=2, pad_value=1.0)
    ax.imshow(grid.permute(1, 2, 0))
    ax.set_title(style.replace("_", " "))
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Colors of 5,000 random paintings: per-channel histograms, mean image, per-pixel std
rng = np.random.default_rng(0)
x = train.images[rng.choice(len(train), min(5000, len(train)), replace=False)].float() / 255
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for channel, color in enumerate(("red", "green", "blue")):
    axes[0].hist(x[:, channel].flatten().numpy(), bins=100, histtype="step", color=color)
axes[0].set_title("pixel values per channel")
axes[1].imshow(x.mean(0).permute(1, 2, 0))
axes[1].set_title("mean painting")
axes[2].imshow(x.std(0).mean(0), cmap="magma")
axes[2].set_title("per-pixel std")
for ax in axes[1:]:
    ax.axis("off")
plt.show()

In [ ]:
# Original sizes (classes.csv in the zip): how much does the center crop remove?
with zipfile.ZipFile(archive_path()) as zf, zf.open("classes.csv") as raw:
    rows = list(csv.DictReader(io.TextIOWrapper(raw, encoding="utf-8")))
width = np.array([int(r["width"]) for r in rows])
height = np.array([int(r["height"]) for r in rows])
aspect = width / height
removed = 1 - np.minimum(width, height) / np.maximum(
    width, height
)  # share of the long side cut away
print(f"width  5/50/95%: {np.percentile(width, [5, 50, 95])}")
print(f"height 5/50/95%: {np.percentile(height, [5, 50, 95])}")
print(f"aspect ratio (w/h) 5/50/95%: {np.percentile(aspect, [5, 50, 95]).round(2)}")
print(f"long side cut away: median {np.median(removed):.0%}, 95% {np.percentile(removed, 95):.0%}")

## Notes for the report

- Split sizes and style balance:
- What the center crop removes:
- What the colors suggest (for example for the generator's last layer):